# 02: country macro and fundamental data pull (run on a Bloomberg terminal)

The FX pulls in `00` and `00b` are market prices: spot, forwards, vol, rates, indices. This notebook pulls the other half of the fundamental picture, the macro releases that "A Foreign Exchange Primer" (Shamah 2008) names in chapter 25 (Fundamental Analysis) and chapter 26 (Key Factors Impacting Currencies) as the forces behind a currency. It answers the supervisors' question directly: how does CPI, or growth, or the trade balance, move a currency.

Run it on a machine with a live Terminal session. It writes one long format parquet, `macro_monthly.parquet`, into `data/raw/`, and leaves the DVC step for you to run once at the bottom.

### Binds with the FX parquets

Same long format as every other pull, `ticker, date, field, value`, one field `PX_LAST` (macro series have no bid or ask). Each ticker decodes back to a `(currency, indicator)` pair through the `parse_macro_ticker` helper defined below, the same way the rate file decodes through `parse_rate_ticker`. Nothing here touches the FX files or the existing loaders.

### Read before running

Every ticker below is an unverified best guess. Country macro tickers are more idiosyncratic than even the EM rate tickers, and none were resolved against a terminal on the authoring machine. The USD block is the most reliable; `cpi_yoy`, `equity` and `gov10y` are the most reliable rows across countries; the growth, labour and external rows for the non US countries are the least certain. Run Step 0 first, keep what resolves, and fix the rest on the terminal. No number here is real until you run the pull and read the probe.

> **This notebook needs a live Bloomberg Terminal, and it has never been run.**
>
> Two separate cautions, and the second is the important one.
>
> `xbbg` talks to a local BLPAPI session, so nothing here runs on a machine without a
> Terminal. That is the same situation as `00b`.
>
> Beyond that, no cell in this notebook has ever produced output, and no macro series
> of any kind exists in `data/raw`. Every ticker in `reference.MACRO_TICKERS` was
> written from Bloomberg's naming conventions rather than read off a screen, so the
> catalogue is a starting point for a Terminal session rather than a table this project
> has ever pulled. Step 0 below is the cell that settles it: run the probe first and
> record which tickers come back EMPTY or FAIL before running the pull.
>
> The code is current with the class API. The three helpers that used to walk the macro
> tables (`macro_tickers`, `parse_macro_ticker`, `macro_lag_months`) were dropped in the
> library rebuild because nothing in the library called them, so they are defined here
> and stay here until a real caller appears.

## How this lines up with the FX data

The macro block is keyed to the same ISO currency codes as the FX universe, one economic block
per country, because every fundamental theory in chapter 25 is relative. The usable signal is
always a differential, home versus foreign, so the two countries have to sit side by side.

Which indicator feeds which of the book's four theories:

| Theory (section) | Mechanism | Indicators here |
|---|---|---|
| Purchasing power parity (25.1) | relative inflation moves the real exchange rate | `cpi_yoy`, `cpi_idx`, `ppi_yoy` |
| Interest rate parity (25.2) | the rate differential offsets expected FX change | already pulled: short rates, plus `gov10y` |
| Balance of payments (25.3) | a trade or current account deficit weakens the currency | `trade_bal`, `curr_acct`, `reserves` |
| Asset market model (25.4) | capital flows dominate; FX tracks equities | `equity`, `gov10y` |

The growth and labour releases (`gdp_yoy`, `ip_yoy`, `retail_yoy`, `unemp`, `pmi`, and the US
only `payroll`, `durables`, `housing`, `leading`) are the ones chapter 25.5 and chapter 26 flag
as market movers, because they shift rate expectations. They are the short term "why" behind a
currency's moves.

### Three alignment rules

1. Country to currency map. `reference.MACRO_TICKERS` is keyed by ISO code, so a macro row
   lands on the right FX column.
2. Monthly frequency. The FX work snaps to month end (`reference.DEFAULT_FREQ` is `"M"`, and
   the loaders take a `freq` argument). We pull macro at monthly periodicity and forward fill
   the quarterly series, GDP and the current account, across the intervening months.
3. Publication lag, the part that is easy to get wrong. Macro is published after the period it
   describes: CPI for month t prints two to four weeks into month t+1. A signal formed at the
   end of month t must only see releases dated on or before t, so we shift each series forward
   by its publication lag before joining. Skipping this is look-ahead bias, and it is exactly
   what the strategy literature is careful about (see `docs/notes/macro-and-fx.md`). The lag
   per indicator is the fourth field of a `reference.MACRO_INDICATORS` entry, applied by the
   `macro_lag_months` helper defined in this notebook.

## The Bloomberg limits, and how this respects them

Macro is cheap. The whole pull is about 130 unique securities at monthly periodicity, a small fraction of any cap.

* Monthly, about 5000 to 7000 unique securities. This pull is around 130, near 2 percent.
* Daily, 500000 hits (one request for one security and field). One field across 130 securities is trivial.
* It is a shared terminal, so still coordinate if teammates are pulling heavily.

If you see `#N/A Limit`, split the batch and resume. But for a pull this small that is unlikely.

In [ ]:
from collections import Counter
from pathlib import Path

import pandas as pd
from xbbg import blp

from fxcarry import Catalog, reference

DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# File names and the pull window belong to the pull, not to the library.
START = "1990-01-01"        # macro histories start later; empty before availability
END = pd.Timestamp.today().strftime("%Y-%m-%d")
MACRO_FILE = "macro_monthly.parquet"
SPOT_FILE = "spot_daily.parquet"
CATALOG = Catalog.default()

# The rebuild kept the macro tables in `reference` but not the three helpers that
# walked them, because nothing in the library calls them. They live here instead,
# and move into the library only if a real caller appears.
_BY_TICKER = {tk: (ccy, ind)
              for ccy, block in reference.MACRO_TICKERS.items()
              for ind, tk in block.items()}


def macro_tickers():
    """Every catalogued macro ticker, deduplicated, in catalogue order."""
    return list(dict.fromkeys(_BY_TICKER))


def parse_macro_ticker(ticker):
    """(currency, indicator) for a catalogued macro ticker, or None if unknown."""
    return _BY_TICKER.get(ticker)


def macro_lag_months(indicator):
    """Publication lag in months: the fourth field of a MACRO_INDICATORS entry."""
    return reference.MACRO_INDICATORS[indicator][3]


tickers = macro_tickers()
print(f"Window:     {START} -> {END}")
print(f"Countries:  {len(reference.MACRO_TICKERS)}")
print(f"Indicators: {len(reference.MACRO_INDICATORS)} in the catalogue")
print(f"Tickers:    {len(tickers)} (partial blocks; a country only lists what it has)")

# Coverage of the catalogue: how many countries carry each indicator
cov = Counter(ind for blk in reference.MACRO_TICKERS.values() for ind in blk)
for ind, (label, freq, tf, lag) in reference.MACRO_INDICATORS.items():
    print(f"  {ind:11s} n={cov.get(ind, 0):2d}  freq={freq} lag={lag}m  {label}")

## Step 0. Validate the tickers cheaply first

The same discipline as the rate probe in `00b`. Pull each macro ticker on its own and print whether it resolved, its start date, and its last value. Read this before trusting anything downstream, then fix the failures on the terminal.

How to find the right ticker when one fails: `ECST <GO>` opens a country's economic statistics tree, `ECO <GO>` and `WECO <GO>` open the release calendars, and for the Markit PMIs the family is `MPMI<country>MA Index`. The two letter Bloomberg country prefixes used below are US, EC (euro area), JN (Japan), UK, SZ (Switzerland), CA, AU, NZ, SW (Sweden), NO, DN (Denmark), MX, SA (South Africa), KO (Korea), SI (Singapore), CZ, HU, PO (Poland), TW, IN.

In [ ]:
def probe(ticker):
    try:
        df = blp.bdh(tickers=ticker, flds=[reference.PX_LAST], start_date="1990-01-01",
                     end_date=END, Per="M", backend="pandas")
        v = df["value"].dropna() if "value" in df.columns else pd.Series(dtype=float)
        if len(v):
            start = str(df["date"].min())[:7] if "date" in df.columns else "?"
            return ("OK   ", len(v), start, round(float(v.iloc[-1]), 3))
        return ("EMPTY", 0, "-", None)
    except Exception as e:
        return ("FAIL ", 0, "-", str(e).splitlines()[-1][:30])

# This is the cell that decides whether the catalogue is real. Every ticker in
# `reference.MACRO_TICKERS` was written from Bloomberg's naming conventions rather
# than read off a screen, so treat an EMPTY or FAIL as the expected outcome for
# some of them and record which.
for ccy, block in reference.MACRO_TICKERS.items():
    for ind, tk in block.items():
        status, n, start, last = probe(tk)
        print(f"{ccy} {ind:11s} {tk:16s} {status} n={n:4d} start={start} last={last}")

## The pull

One batched monthly pull across every macro ticker, written as a single long frame. Monthly periodicity (`Per="M"`) lines the file up with the FX month end panel; quarterly series come back at their quarter end dates and are forward filled at alignment time.

In [ ]:
def bdh_batched(tickers, flds, start=START, end=END, per="M", batch=100):
    '''Pull a list in chunks and return one concatenated long frame. Monthly by
    default, since macro is a monthly or quarterly series.'''
    tickers = list(dict.fromkeys(tickers))            # dedupe, keep order
    frames = []
    for i in range(0, len(tickers), batch):
        chunk = tickers[i : i + batch]
        frames.append(blp.bdh(tickers=chunk, flds=flds, start_date=start,
                              end_date=end, Per=per, backend="pandas"))
        print(f"  {i + len(chunk):4d}/{len(tickers)} pulled")
    return pd.concat(frames, ignore_index=True)

macro = bdh_batched(macro_tickers(), [reference.PX_LAST])
macro.to_parquet(DATA_DIR / MACRO_FILE)
print(f"Macro: {macro.shape} -> {MACRO_FILE}")
print(f"resolved tickers: {macro['ticker'].nunique()} / {len(macro_tickers())}")

## Aligning macro to the FX panel, point in time

This is the "make sure it aligns with FX" step. It runs after the pull, on the same terminal, using the macro file just written and the local FX spot parquet. Two small helpers: `decode_macro` turns the long frame into `(currency, indicator)` rows, and `macro_panel` builds a dates by currency panel for one indicator, forward filled and shifted by its publication lag so it is point in time.

In [ ]:
def coverage(frame):
    """First and last valid index and observation count, per column.

    `Quotes.coverage` does this for two-sided market data. A macro panel has one
    side, so this is the same summary for a plain frame.
    """
    return pd.DataFrame(
        {col: {"first_valid": s.first_valid_index(),
               "last_valid": s.last_valid_index(),
               "n_obs": int(s.notna().sum())}
         for col, s in frame.items()}).T


def decode_macro(macro_long):
    '''Long macro frame -> same frame with ccy and indicator columns decoded
    from the ticker (drops any ticker not in the map).'''
    m = macro_long.copy()
    keys = m["ticker"].map(parse_macro_ticker)
    m = m[keys.notna()].copy()
    m["ccy"] = [k[0] for k in keys.dropna()]
    m["indicator"] = [k[1] for k in keys.dropna()]
    m["date"] = pd.to_datetime(m["date"])
    return m


def macro_panel(macro_decoded, indicator, lag=True):
    '''Dates (monthly Period) by currency panel for one indicator. Forward
    filled so quarterly series cover the months between releases, then shifted
    forward by the indicator's publication lag so a month t signal only sees
    what had printed by t.'''
    sub = macro_decoded[(macro_decoded["indicator"] == indicator)
                        & (macro_decoded["field"] == reference.PX_LAST)].copy()
    sub["month"] = sub["date"].dt.to_period("M")
    wide = sub.pivot_table(index="month", columns="ccy", values="value").sort_index()
    wide = wide.ffill()
    if lag:
        wide = wide.shift(macro_lag_months(indicator))
    return wide

macro = pd.read_parquet(DATA_DIR / MACRO_FILE)
dec = decode_macro(macro)
print("decoded rows:", len(dec), "| currencies:", dec["ccy"].nunique(),
      "| indicators:", sorted(dec["indicator"].unique()))

In [ ]:
# Example: the PPP signal. Home inflation minus US inflation, point in time,
# joined onto the FX month end grid so it sits next to spot and forwards.
from fxcarry import ParquetSource

cpi = macro_panel(dec, "cpi_yoy", lag=True)
infl_diff = cpi.sub(cpi["USD"], axis=0)                 # home CPI minus US CPI

spot = ParquetSource(DATA_DIR / SPOT_FILE).quotes(
    CATALOG.label_map("spot"), freq="M").mid
fx_months = spot.index.to_period("M")                   # the FX month end grid
aligned = infl_diff.reindex(fx_months)                  # macro dropped onto that grid

print("FX month grid:", fx_months.min(), "->", fx_months.max(), "| n =", len(fx_months))
print("inflation differential coverage (non-empty months):",
      int(aligned.notna().any(axis=1).sum()))
display(coverage(aligned))

## Track with DVC, run once in a shell

Start the rclone bridge, add the one new parquet, commit its pointer, and push to Box. The bridge command also lives in the root `README.md`.

```bash
rclone serve webdav uchicago-box:fxcarry-data --addr 127.0.0.1:8080 --vfs-cache-mode writes

cd fxcarry
dvc add data/raw/macro_monthly.parquet
git add data/raw/macro_monthly.parquet.dvc
git commit -m "data: country macro and fundamental indicators pull (Primer ch. 25-26)"
dvc push
```

Then on the other machine, `git pull` and `dvc pull` bring it down.

## What next

- The research on how macro moves FX, and how strategy papers treat it, is written up in
  `docs/notes/macro-and-fx.md`. That note is the "how does CPI move a currency" defense answer
  and feeds the per currency blocks.
- `m2` (money supply) and `reserves` (official FX reserves) are populated best effort and are
  the least certain rows, because countries headline different money aggregates and report
  reserves on different conventions. Confirm them first at Step 0. The chapter 26 US releases
  (`payroll`, `durables`, `housing`, `leading`) are populated only for USD.
- Nothing here belongs in the library yet. The three helpers at the top and the two panel
  functions above are enough to align the file with the FX panel, and they stay in this
  notebook until something in the pipeline actually calls them. If a macro signal ever earns a
  place in a strategy, that is the moment to promote them, not before.